# Notebook 10 Part 4: Data Skew and Skew Handling in PySpark



## 1. What Is Data Skew?

Data skew means:

```text
Data is not distributed evenly across partitions.
```

Example:

```text
Delhi   → 90,000 rows
Mumbai  → 4,000 rows
Pune    → 3,000 rows
Chennai → 3,000 rows
```

One key contains most of the data, so one partition may receive much more work.

## 2. Why Is Skew a Problem?

Suppose Spark creates four tasks:

```text
Task 1 → 90,000 rows
Task 2 → 4,000 rows
Task 3 → 3,000 rows
Task 4 → 3,000 rows
```

Tasks 2, 3 and 4 finish quickly.

Task 1 takes much longer.

The whole stage waits for the slowest task.

## 3. Simple Rule

```text
Balanced partitions → similar work
Skewed partitions   → one/few partitions have much more work
```

## 4. Windows Local Setup

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 5. Create SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Data_Skew_in_PySpark") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

## 6. Create a Skewed Dataset

Most rows will belong to `Delhi`.

In [3]:
data = []

for i in range(1, 100001):
    if i <= 90000:
        city = "Delhi"
    elif i <= 94000:
        city = "Mumbai"
    elif i <= 97000:
        city = "Pune"
    else:
        city = "Chennai"

    amount = 1000 + (i % 100) * 10
    data.append((i, city, amount))

sales_df = spark.createDataFrame(
    data,
    ["order_id", "city", "amount"]
)

sales_df.show(5)
print("Total rows:", sales_df.count())

+--------+-----+------+
|order_id| city|amount|
+--------+-----+------+
|       1|Delhi|  1010|
|       2|Delhi|  1020|
|       3|Delhi|  1030|
|       4|Delhi|  1040|
|       5|Delhi|  1050|
+--------+-----+------+
only showing top 5 rows
Total rows: 100000


## 7. Detect Skew Using Key Counts

One easy method:

```python
groupBy().count()
```

In [5]:
sales_df.groupBy("city") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+-------+-----+
|   city|count|
+-------+-----+
|  Delhi|90000|
| Mumbai| 4000|
|Chennai| 3000|
|   Pune| 3000|
+-------+-----+



## 8. Check Number of Partitions

In [6]:
print("Current partitions:", sales_df.rdd.getNumPartitions())

Current partitions: 1


The number of partitions alone does not tell us whether data is balanced.

We also need to inspect rows per partition.

## 9. Inspect Rows Per Partition

In [7]:
from pyspark.sql.functions import spark_partition_id

sales_df.groupBy(
    spark_partition_id().alias("partition_id")
).count().show()

+------------+------+
|partition_id| count|
+------------+------+
|           0|100000|
+------------+------+



This shows:

```text
Partition ID → Row count
```

## 10. Repartition by the Skewed Key

In [11]:
city_partitioned_df = sales_df.repartition(4, "city")

print("Partitions:", city_partitioned_df.rdd.getNumPartitions())

Partitions: 4


In [12]:
city_partitioned_df.groupBy(
    spark_partition_id().alias("partition_id")
).count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3000|
|           1|90000|
|           3| 7000|
+------------+-----+



Important:

```text
repartition() does not automatically solve key skew.
```

If the key itself is skewed, one partition can still become very large.

# 15. Solution: Salting

If both join tables are large and broadcast is not possible, one common technique is:

```text
Salting
```

Salting adds an extra value to a hot key so that the key can be spread into multiple smaller keys.

## 16. Salting Concept

Before:

```text
Delhi
Delhi
Delhi
Delhi
Delhi
Delhi
```

After:

```text
Delhi_0
Delhi_1
Delhi_2
Delhi_3
Delhi_0
Delhi_1
```

## 17. Add Salt to the Large DataFrame

In [13]:
hot_city = "Delhi"
salt_buckets = 4

print("Hot key:", hot_city)
print("Salt buckets:", salt_buckets)

Hot key: Delhi
Salt buckets: 4


use `uniform()` to directly generate random integer salt values.

Only Delhi gets random salt values.

Other cities get salt `0`.

In [15]:
from pyspark.sql.functions import col, when, lit, uniform, concat_ws

salted_sales_df = sales_df.withColumn(
    "salt",
    when(
        col("city") == hot_city,
        uniform(0, salt_buckets, seed=42)
    ).otherwise(lit(0))
)

salted_sales_df.select(
    "order_id", "city", "amount", "salt"
).show(15)

+--------+-----+------+----+
|order_id| city|amount|salt|
+--------+-----+------+----+
|       1|Delhi|  1010|   2|
|       2|Delhi|  1020|   2|
|       3|Delhi|  1030|   3|
|       4|Delhi|  1040|   1|
|       5|Delhi|  1050|   2|
|       6|Delhi|  1060|   2|
|       7|Delhi|  1070|   3|
|       8|Delhi|  1080|   0|
|       9|Delhi|  1090|   3|
|      10|Delhi|  1100|   3|
|      11|Delhi|  1110|   1|
|      12|Delhi|  1120|   2|
|      13|Delhi|  1130|   1|
|      14|Delhi|  1140|   3|
|      15|Delhi|  1150|   2|
+--------+-----+------+----+
only showing top 15 rows


Now Delhi is split into:

```text
Delhi_0
Delhi_1
Delhi_2
Delhi_3
```

This can distribute work more evenly.

## 21. Step 5: Create Salt Values

In [25]:
df_updated = salted_sales_df.withColumn("city_salted", concat_ws("_", col("city"), col("salt")))
df_updated.show()

+--------+-----+------+----+-----------+
|order_id| city|amount|salt|city_salted|
+--------+-----+------+----+-----------+
|       1|Delhi|  1010|   2|    Delhi_2|
|       2|Delhi|  1020|   2|    Delhi_2|
|       3|Delhi|  1030|   3|    Delhi_3|
|       4|Delhi|  1040|   1|    Delhi_1|
|       5|Delhi|  1050|   2|    Delhi_2|
|       6|Delhi|  1060|   2|    Delhi_2|
|       7|Delhi|  1070|   3|    Delhi_3|
|       8|Delhi|  1080|   0|    Delhi_0|
|       9|Delhi|  1090|   3|    Delhi_3|
|      10|Delhi|  1100|   3|    Delhi_3|
|      11|Delhi|  1110|   1|    Delhi_1|
|      12|Delhi|  1120|   2|    Delhi_2|
|      13|Delhi|  1130|   1|    Delhi_1|
|      14|Delhi|  1140|   3|    Delhi_3|
|      15|Delhi|  1150|   2|    Delhi_2|
|      16|Delhi|  1160|   2|    Delhi_2|
|      17|Delhi|  1170|   0|    Delhi_0|
|      18|Delhi|  1180|   2|    Delhi_2|
|      19|Delhi|  1190|   0|    Delhi_0|
|      20|Delhi|  1200|   3|    Delhi_3|
+--------+-----+------+----+-----------+
only showing top

In [27]:
city_partitioned_df = df_updated.repartition(7,"city_salted")

print("Partitions:", city_partitioned_df.rdd.getNumPartitions())

Partitions: 7


In [30]:
city_partitioned_df.groupBy(
    spark_partition_id().alias("partition_id"),"city_salted"
).count().show()

+------------+-----------+-----+
|partition_id|city_salted|count|
+------------+-----------+-----+
|           0|    Delhi_2|22366|
|           4|    Delhi_0|22588|
|           4|     Pune_0| 3000|
|           5|    Delhi_3|22514|
|           5|   Mumbai_0| 4000|
|           6|    Delhi_1|22532|
|           6|  Chennai_0| 3000|
+------------+-----------+-----+



## 20. Salting Has a Cost

Salting adds complexity:

```text
Extra salt column
Duplicate rows on the other side
More data volume
More complex code
```

Try simpler solutions first when possible.

# 21. Solution 3: Adaptive Query Execution (AQE)

AQE allows Spark to optimize execution while the query is running.

Important configuration:

```text
spark.sql.adaptive.enabled
```

In [23]:
spark.conf.get("spark.sql.adaptive.enabled")

'true'

## 22. Check AQE Skew Join Setting

In [24]:
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")

'true'

When enabled, Spark can detect abnormally large shuffle partitions during eligible joins and split them into smaller pieces.

## 23. Enable AQE and Skew Join Optimization

In [25]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Skew Join:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

AQE: true
Skew Join: true


AQE can help, but you should still check:

```text
Key distribution
Spark UI
Execution plan
Partition sizes
```

# 24. How to Detect Skew in Real Projects

Common signs:

```text
1. One task takes much longer than others
2. One partition is much larger
3. One key appears extremely often
4. A stage waits for one/few tasks
5. High shuffle read/write for a few tasks
```

## 25. Spark UI and Skew

Example:

```text
Task 1 → 4 sec
Task 2 → 5 sec
Task 3 → 3 sec
Task 4 → 80 sec
```

One task being dramatically slower can indicate skew.

In [26]:
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark UI: http://127.0.0.1:4040


In [ ]:
spark.stop()